# Ampliación opcional · CLIP en vivo

No forma parte obligatoria de las 150 min. Usa el modelo real para frases nuevas y para recalcular la galería. Descarga pesos de varios cientos de MB. Funciona con CPU; GPU opcional. No necesita clave de API para el modelo público. Revisa condiciones de uso en su tarjeta.

La instalación afecta al entorno donde ejecutes este notebook: usa Colab o un entorno local separado. Si se solicita reiniciar el entorno, hazlo antes de continuar. El notebook principal no necesita estas dependencias.

La revisión exacta, el procesador lento y el identificador del modelo se conservan en el archivo de procedencia. No se activan scripts remotos del repositorio del modelo.

In [ ]:
# Ejecutar solo en el entorno dedicado a esta ampliación.
%pip install -q "torch>=2.6,<3" "transformers==4.57.6"

In [ ]:
from pathlib import Path
import json, hashlib
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE_URL = "https://raw.githubusercontent.com/afalco/vision-ia-producto/main/05_multimodal_generativa/recursos/"
opciones = [Path("recursos"), Path("05_multimodal_generativa/recursos"),
            Path("vision-ia-producto/05_multimodal_generativa/recursos")]
RECURSOS = next((p for p in opciones if (p/"catalogo.json").exists()), Path("recursos_tema5"))
RECURSOS.mkdir(parents=True, exist_ok=True)
def recurso(nombre):
    destino = RECURSOS / nombre
    if not destino.exists():
        print("Descargando recurso:", nombre)
        try:
            with urlopen(BASE_URL + nombre, timeout=60) as respuesta:
                destino.write_bytes(respuesta.read())
        except Exception as error:
            raise RuntimeError("No se pudo descargar " + nombre +
                ". Comprueba que los cambios del tema 5 estén publicados o descarga el repositorio completo.") from error
    return destino

catalogo = json.loads(recurso("catalogo.json").read_text(encoding="utf-8"))
procedencia = json.loads(recurso("procedencia_clip.json").read_text(encoding="utf-8"))
for nombre, esperado in procedencia["sha256_archivos"].items():
    actual = hashlib.sha256(recurso(nombre).read_bytes()).hexdigest()
    if actual != esperado:
        raise ValueError("Recurso cambiado o incompleto: " + nombre + ". Descarga una versión coherente del tema.")
consultas = catalogo["consultas"]
nombres = [e["id"] for e in catalogo["imagenes"]]
imagenes = [Image.open(recurso(e["archivo"])).convert("RGB") for e in catalogo["imagenes"]]
with np.load(recurso("clip_vectores.npz"), allow_pickle=False) as datos:
    vectores_imagen = datos["imagenes"].copy()
    vectores_texto = datos["textos"].copy()
    escala_logits = float(datos["escala"])
assert vectores_imagen.shape == (len(imagenes), procedencia["vector_dim"])
assert vectores_texto.shape == (len(consultas), procedencia["vector_dim"])
assert np.allclose(np.linalg.norm(vectores_imagen, axis=1), 1, atol=1e-5)
assert np.allclose(np.linalg.norm(vectores_texto, axis=1), 1, atol=1e-5)
print("Modelo:", procedencia["modelo"], "Revisión:", procedencia["revision"])
print("Galería:", len(imagenes), "imágenes; banco:", len(consultas), "consultas")

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
MODELO = procedencia["modelo"]
REVISION = procedencia["revision"]
modelo = CLIPModel.from_pretrained(MODELO, revision=REVISION).eval()
procesador = CLIPProcessor.from_pretrained(MODELO, revision=REVISION, use_fast=False)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo = modelo.to(dispositivo)
print("Dispositivo:", dispositivo)
# Limitar hilos evita que equipos pequeños dediquen todos sus recursos al ejemplo.
if dispositivo == "cpu": torch.set_num_threads(2)

In [ ]:
# Recalcula los vectores de imagen; ya no dependemos de sus resultados guardados.
entradas = procesador(images=imagenes, return_tensors="pt").to(dispositivo)
with torch.inference_mode():
    vi = modelo.get_image_features(**entradas)
    vi = vi / vi.norm(dim=-1, keepdim=True)
vi = vi.cpu().numpy()
print("Forma de vectores:", vi.shape)
print("Máxima diferencia con el cache:", float(np.max(np.abs(vi-vectores_imagen))))
CONSULTA_NUEVA = "a blue mug on a white background"
entradas = procesador(text=[CONSULTA_NUEVA], return_tensors="pt", padding=True).to(dispositivo)
with torch.inference_mode():
    vt = modelo.get_text_features(**entradas)
    vt = vt / vt.norm(dim=-1, keepdim=True)
scores = (vi @ vt.cpu().numpy().T).ravel()
ids = np.argsort(-scores)[:3]
print(pd.DataFrame({"Imagen": [nombres[i] for i in ids], "Coseno": scores[ids]}).round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, i in zip(axes, ids):
    ax.imshow(imagenes[i])
    ax.set_title(f"{nombres[i]} · {scores[i]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Experimento

Prueba una frase nueva cada vez. Documenta predicción, ranking y limitación observada. Usa inglés como caso principal de este modelo. Una frase no incluida en el banco ya se calcula aquí con el codificador de texto real. No se ha entrenado un modelo nuevo y no se generan imágenes.